# 08 — OSM South Carolina Power Network

This notebook does one bounded job:

1. download South Carolina `power=line` + `power=minor_line` from OpenStreetMap once;
2. cache the raw Overpass JSON;
3. parse real OSM ways and node IDs;
4. build 33/46/115/230 kV graph views;
5. test the 33/46 kV assets that HIFLD cannot route;
6. tag Dominion-internal endpoint/asset names as `requires_primary_source` instead of endlessly tuning fuzzy matching.

Important: existing-line routes must come from traced public geometry. This notebook never creates a straight-line substitute.

Before running it, put `osm_power_network.py` at:

```text
src/gridlock/osm_power_network.py
```


In [2]:
from pathlib import Path
import json
import re
import sys

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

from gridlock.paths import (
    PROJECT_ROOT,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
)

from gridlock.osm_power_network import (
    download_osm_power_network,
    load_osm_power_network,
    voltage_summary,
    build_voltage_graph,
    graph_summary,
    route_between_points,
    degree_table,
)

OSM_RAW = RAW_DATA_DIR / "osm" / "SC" / "power_lines_statewide_overpass.json"

ROUTE_RESULTS_CSV = (
    PROCESSED_DATA_DIR
    / "dominion"
    / "dominion_route_resolution_v3.csv"
)

ENDPOINT_REGISTRY_CSV = (
    PROCESSED_DATA_DIR
    / "dominion"
    / "dominion_endpoint_registry.csv"
)

TAGGED_ROUTE_RESULTS_CSV = (
    PROCESSED_DATA_DIR
    / "dominion"
    / "dominion_route_resolution_v3_tagged.csv"
)

PRIMARY_SOURCE_QUEUE_CSV = (
    PROCESSED_DATA_DIR
    / "dominion"
    / "dominion_requires_primary_source.csv"
)

OSM_ROUTE_OUTPUT_CSV = (
    PROCESSED_DATA_DIR
    / "dominion"
    / "dominion_osm_low_voltage_routes.csv"
)

OSM_ROUTE_OUTPUT_GEOJSON = (
    PROCESSED_DATA_DIR
    / "geo"
    / "dominion_osm_low_voltage_routes.geojson"
)

for path in [
    TAGGED_ROUTE_RESULTS_CSV,
    PRIMARY_SOURCE_QUEUE_CSV,
    OSM_ROUTE_OUTPUT_CSV,
    OSM_ROUTE_OUTPUT_GEOJSON,
]:
    path.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("OSM cache:", OSM_RAW)


Project root: C:\Users\sebas\PycharmProjects\Git\Project_secret_project
OSM cache: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\osm\SC\power_lines_statewide_overpass.json


## 1. Download once, then reuse the cache

This is deliberately **one statewide query**, not one query per voltage.

OSM voltage tags can be missing or multi-valued. We do not want the download query itself to discard useful geometry.


In [3]:
raw_path = download_osm_power_network(
    OSM_RAW,
    state_iso="US-SC",
    refresh=False,
)

print("Using:", raw_path)


Using: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\osm\SC\power_lines_statewide_overpass.json


## 2. Parse OSM ways and actual node IDs

A visual crossing is **not** treated as an electrical junction.

Two line segments connect only when they share an actual OSM node ID.


In [4]:
ways, nodes = load_osm_power_network(OSM_RAW)

print("OSM power ways:", len(ways))
print("OSM nodes:", len(nodes))

display(
    ways[
        [
            "way_id",
            "power",
            "name",
            "operator",
            "voltage_raw",
            "voltages_kv",
            "circuits",
        ]
    ].head(20)
)


OSM power ways: 4963
OSM nodes: 113233


,way_id,power,name,operator,voltage_raw,voltages_kv,circuits
0,8151871,line,Seneca - Central #1/2 100kV,NaN,100000,[100.0],2
1,9142560,line,NaN,NaN,115000,[115.0],2
2,12077208,line,NaN,NaN,44000,[44.0],2
3,12077320,line,NaN,NaN,44000,[44.0],NaN
4,12077401,line,NaN,NaN,44000,[44.0],NaN
5,12080662,line,NaN,NaN,115000,[115.0],NaN
6,12081739,line,NaN,NaN,NaN,[],NaN
7,12083596,line,NaN,NaN,115000,[115.0],NaN
8,12087069,line,Urquhart - Fairfax 115kV,NaN,115000,[115.0],NaN
9,12105323,line,NaN,NaN,NaN,[],NaN


## 3. What voltage coverage did OSM actually give us?

In [5]:
coverage = voltage_summary(ways)
display(coverage)


,voltage_kv,ways
0,10,2
1,12.47,44
2,24,2
3,33,30
4,34.5,48
5,44,650
6,46,167
7,69,130
8,100,1223
9,115,1010


In [6]:
for voltage in [33, 44, 46]:
    subset = ways[
        ways["voltages_kv"].apply(
            lambda values: voltage in values
        )
    ]

    print(f"\n--- {voltage} kV ---")
    print("ways:", len(subset))

    display(
        subset[
            [
                "way_id",
                "name",
                "operator",
                "owner",
                "voltage_raw",
                "circuits",
                "power",
            ]
        ].head(50)
    )

    print("\nOperators:")
    display(
        subset["operator"]
        .fillna("UNKNOWN")
        .value_counts()
        .head(20)
        .to_frame("ways")
    )


--- 33 kV ---
ways: 30


,way_id,name,operator,owner,voltage_raw,circuits,power
922,648480645,NaN,NaN,None,33000,2,line
2057,1077742256,NaN,NaN,None,33000,NaN,line
2058,1077742257,NaN,NaN,None,33000,NaN,line
2059,1077742258,NaN,NaN,None,33000,NaN,line
2060,1077742260,NaN,NaN,None,33000,NaN,line
2061,1077742261,NaN,NaN,None,33000,NaN,line
2140,1078248501,NaN,NaN,None,115000;33000,2,line
2167,1078308097,NaN,NaN,None,33000,NaN,line
2168,1078308098,NaN,NaN,None,33000,NaN,line
2169,1078308099,NaN,NaN,None,33000,NaN,line



Operators:


,ways
operator,
UNKNOWN,30



--- 44 kV ---
ways: 650


...

,ways
operator,
UNKNOWN,596
Duke Energy,54



--- 46 kV ---
ways: 167


,way_id,name,operator,owner,voltage_raw,circuits,power
15,12139895,NaN,NaN,None,46000,NaN,line
398,185577982,NaN,NaN,None,230000;46000,2,line
461,236469792,NaN,NaN,None,46000,NaN,line
603,380872162,NaN,NaN,None,46000,NaN,line
611,405874589,NaN,NaN,None,46000,NaN,line
612,405881970,NaN,NaN,None,46000,NaN,line
914,623292013,NaN,NaN,None,46000,NaN,line
954,722704904,NaN,NaN,None,46000,NaN,line
1177,1041994615,NaN,NaN,None,46000,NaN,line
1349,1065939219,NaN,South Carolina Gas & Electric,None,46000,NaN,line



Operators:


,ways
operator,
UNKNOWN,158
South Carolina Gas & Electric,6
South Carolina Electric & Gas,2
Dominion Energy South Carolina,1


## 4. Build voltage-specific graph views

Unknown-voltage edges are excluded from the trusted graph.

We also build a `with_unknown_voltage` graph. If a route only works there, the route is `review_required` instead of silently trusted.


In [9]:
TARGET_VOLTAGES = [33, 46, 115, 230]

graphs = {}
graphs_review = {}
summary_rows = []

for voltage in TARGET_VOLTAGES:
    trusted = build_voltage_graph(
        ways,
        nodes,
        voltage,
        include_unknown_voltage=False,
    )

    review = build_voltage_graph(
        ways,
        nodes,
        voltage,
        include_unknown_voltage=True,
    )

    graphs[voltage] = trusted
    graphs_review[voltage] = review

    trusted_summary = graph_summary(trusted)
    trusted_summary["graph"] = "trusted"
    summary_rows.append(trusted_summary)

    review_summary = graph_summary(review)
    review_summary["graph"] = "with_unknown_voltage"
    summary_rows.append(review_summary)

graph_summaries = pd.DataFrame(summary_rows)

display(
    graph_summaries[
        [
            "voltage_kv",
            "graph",
            "nodes",
            "edges",
            "ways",
        ]
    ]
)


,voltage_kv,graph,nodes,edges,ways
0,33.0,trusted,663,658,30
1,33.0,with_unknown_voltage,7855,7697,957
2,46.0,trusted,9122,9099,167
3,46.0,with_unknown_voltage,16308,16138,1094
4,115.0,trusted,29966,29970,1010
5,115.0,with_unknown_voltage,37131,37009,1937
6,230.0,trusted,18839,18935,772
7,230.0,with_unknown_voltage,26015,25974,1699


## 5. Tag Dominion-internal identifiers now

These are not generic place-name failures. They should not motivate looser fuzzy matching.

For now, explicitly tag rows containing the internal labels we already know about:

- `Frogmore Transmission`
- `CAE Industrial Park`
- `structure <number>`

The list is deliberately conservative. Add another internal identifier only when the project source gives us a reason to.


In [11]:
route_results = pd.read_csv(ROUTE_RESULTS_CSV)

PRIMARY_SOURCE_EXACT_TERMS = {
    "frogmore transmission": "Dominion/internal facility naming",
    "cae industrial park": "Dominion/internal facility naming",
}

PRIMARY_SOURCE_PATTERNS = [
    (
        re.compile(r"\bstructure\s+\d+\b", re.IGNORECASE),
        "numbered structure requires project/engineering source",
    ),
]


def primary_source_reason(row):
    searchable_columns = [
        "endpoint_a",
        "endpoint_b",
        "asset_name",
        "project_name",
        "review_note",
    ]

    values = []

    for column in searchable_columns:
        value = row.get(column)

        if value is None or pd.isna(value):
            value = ""

        values.append(str(value))

    combined = " | ".join(values)
    combined_lower = combined.lower()
    reasons = []

    for term, reason in PRIMARY_SOURCE_EXACT_TERMS.items():
        if term in combined_lower:
            reasons.append(f"{term}: {reason}")

    for pattern, reason in PRIMARY_SOURCE_PATTERNS:
        if pattern.search(combined):
            reasons.append(f"{pattern.pattern}: {reason}")

    return "; ".join(dict.fromkeys(reasons))


route_results["primary_source_reason"] = route_results.apply(
    primary_source_reason,
    axis=1,
)

route_results["resolution_requirement"] = route_results[
    "primary_source_reason"
].apply(
    lambda value: (
        "requires_primary_source"
        if isinstance(value, str) and value.strip()
        else "standard_public_gis"
    )
)

route_results.to_csv(
    TAGGED_ROUTE_RESULTS_CSV,
    index=False,
)

primary_source_queue = route_results[
    route_results["resolution_requirement"]
    == "requires_primary_source"
].copy()

primary_source_queue.to_csv(
    PRIMARY_SOURCE_QUEUE_CSV,
    index=False,
)

print(
    "Rows tagged requires_primary_source:",
    len(primary_source_queue),
)

display(
    primary_source_queue[
        [
            "project_name",
            "asset_name",
            "endpoint_a",
            "endpoint_b",
            "voltage_kv",
            "route_status",
            "primary_source_reason",
        ]
    ]
)


Rows tagged requires_primary_source: 4


,project_name,asset_name,endpoint_a,endpoint_b,voltage_kv,route_status,primary_source_reason
32,Burton-St Helena 115kV: Rebuild Frogmore Distribution – St. Helena Section,Frogmore Transmission - St Helena,Frogmore Transmission,St Helena,115,endpoint_anchor_missing,frogmore transmission: Dominion/internal facility naming
43,Lyles – Kilbourne Park 115kV: Replace Structure 67 – Kilbourne Park,Lyles - Kilbourne Park structure 67 segment,Lyles,Kilbourne Park,115,endpoint_anchor_missing,\bstructure\s+\d+\b: numbered structure requires project/engineering source
44,Lyles – Kilbourne Park 115kV: Replace Structure 67 – Kilbourne Park,Richland Mall GOAB - Kilbourne Park conductor,Richland Mall GOAB,Kilbourne Park,115,endpoint_anchor_missing,\bstructure\s+\d+\b: numbered structure requires project/engineering source
73,CAE Industrial Park - Springdale Sub 115kV: Replace Laminated Wood Polls.,CAE Industrial Park - Springdale laminated poles,CAE Industrial Park,Springdale,115,endpoint_anchor_missing,cae industrial park: Dominion/internal facility naming


## 6. Focus on HIFLD coverage gaps

These are the rows where HIFLD does not provide a usable Dominion network at the project's voltage.

OSM is now the primary network candidate for these assets.


In [12]:
coverage_gaps = route_results[
    route_results["route_status"]
    == "voltage_not_in_dominion_hifld"
].copy()

print("HIFLD voltage coverage gaps:", len(coverage_gaps))

display(
    coverage_gaps[
        [
            "asset_key",
            "project_name",
            "asset_name",
            "endpoint_a",
            "endpoint_b",
            "voltage_kv",
            "resolution_requirement",
        ]
    ]
)


HIFLD voltage coverage gaps: 11


,asset_key,project_name,asset_name,endpoint_a,endpoint_b,voltage_kv,resolution_requirement
0,SC:scrtp_projects.pdf:10:5392 A-C:1,Coit – Gills Creek 115kV: Construct,Coit - Gills Creek existing 33 kV line,Coit,Gills Creek,33,standard_public_gis
31,SC:scrtp_projects.pdf:20:6809 G:2,Hooks - Modoc 115/46 kV Rebuild,Hooks - Modoc 46 kV,Hooks,Modoc,46,standard_public_gis
36,SC:scrtp_projects.pdf:23:6810 U:1,Cameron Jct – Elloree 46 kV Rebuild,Cameron Jct - Elloree,Cameron Jct,Elloree,46,standard_public_gis
38,SC:scrtp_projects.pdf:25:6810 J:1,Elloree – Santee City 46 kV: Rebuild,Elloree - Santee City,Elloree,Santee City,46,standard_public_gis
39,SC:scrtp_projects.pdf:26:6810 O:1,Urquhart – Aiken PSA 46 kV: Rebuild,Urquhart - Aiken PSA tap point,Urquhart,Aiken PSA,46,standard_public_gis
42,SC:scrtp_projects.pdf:28:6809 I2:1,Adams Run – Red House Road 46 kV: Replace Dawhoo River Crossing and other Structures,Adams Run - Red House Road listed structures,Adams Run,Red House Road,46,standard_public_gis
55,SC:scrtp_projects.pdf:35:6809 F:1,Orangeburg #1 - Cameron Jct 46kV Line Rebuild,Orangeburg #1 - Cameron Jct,Orangeburg #1,Cameron Jct,46,standard_public_gis
69,SC:scrtp_projects.pdf:40:6810 T:1,Cameron Jct – Cameron – St Matthews 46 kV Rebuild,Cameron Jct - Cameron,Cameron Jct,Cameron,46,standard_public_gis
70,SC:scrtp_projects.pdf:40:6810 T:2,Cameron Jct – Cameron – St Matthews 46 kV Rebuild,Cameron - St Matthews,Cameron,St Matthews,46,standard_public_gis
84,SC:scrtp_projects.pdf:48:6809 M:2,Modoc – McCormick 115/46 kV Rebuild,Modoc - McCormick 46 kV,Modoc,McCormick,46,standard_public_gis


## 7. Load trusted endpoint anchors

We route only when both endpoints already have independently resolved coordinates.

This notebook does **not** make up a coordinate for a missing Dominion-internal label.


In [13]:
endpoint_registry = pd.read_csv(ENDPOINT_REGISTRY_CSV)

endpoint_registry["normalized_endpoint"] = (
    endpoint_registry["endpoint_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)


def as_bool(value):
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() == "true"


endpoint_registry["found_bool"] = endpoint_registry["found"].apply(as_bool)


def endpoint_lookup(name, voltage_kv):
    if name is None or pd.isna(name):
        return None

    key = str(name).strip().lower()

    candidates = endpoint_registry[
        (endpoint_registry["normalized_endpoint"] == key)
        & (endpoint_registry["voltage_kv"].astype(float) == float(voltage_kv))
        & (endpoint_registry["found_bool"])
    ]

    if candidates.empty:
        return None

    row = candidates.iloc[0]

    if pd.isna(row["longitude"]) or pd.isna(row["latitude"]):
        return None

    return {
        "name": row.get("resolved_name"),
        "lon": float(row["longitude"]),
        "lat": float(row["latitude"]),
        "source": row.get("source"),
        "feature_id": row.get("feature_id"),
    }


anchor_check_rows = []

for _, asset in coverage_gaps.iterrows():
    voltage = float(asset["voltage_kv"])

    a = endpoint_lookup(asset.get("endpoint_a"), voltage)
    b = endpoint_lookup(asset.get("endpoint_b"), voltage)

    anchor_check_rows.append({
        "asset_key": asset["asset_key"],
        "project_name": asset["project_name"],
        "endpoint_a": asset.get("endpoint_a"),
        "a_found": a is not None,
        "endpoint_b": asset.get("endpoint_b"),
        "b_found": b is not None,
        "voltage_kv": voltage,
        "resolution_requirement": asset.get("resolution_requirement"),
    })

anchor_check = pd.DataFrame(anchor_check_rows)
display(anchor_check)


,asset_key,project_name,endpoint_a,a_found,endpoint_b,b_found,voltage_kv,resolution_requirement
0,SC:scrtp_projects.pdf:10:5392 A-C:1,Coit – Gills Creek 115kV: Construct,Coit,False,Gills Creek,False,33.0,standard_public_gis
1,SC:scrtp_projects.pdf:20:6809 G:2,Hooks - Modoc 115/46 kV Rebuild,Hooks,False,Modoc,True,46.0,standard_public_gis
2,SC:scrtp_projects.pdf:23:6810 U:1,Cameron Jct – Elloree 46 kV Rebuild,Cameron Jct,True,Elloree,True,46.0,standard_public_gis
3,SC:scrtp_projects.pdf:25:6810 J:1,Elloree – Santee City 46 kV: Rebuild,Elloree,True,Santee City,False,46.0,standard_public_gis
4,SC:scrtp_projects.pdf:26:6810 O:1,Urquhart – Aiken PSA 46 kV: Rebuild,Urquhart,True,Aiken PSA,False,46.0,standard_public_gis
5,SC:scrtp_projects.pdf:28:6809 I2:1,Adams Run – Red House Road 46 kV: Replace Dawhoo River Crossing and other Structures,Adams Run,True,Red House Road,False,46.0,standard_public_gis
6,SC:scrtp_projects.pdf:35:6809 F:1,Orangeburg #1 - Cameron Jct 46kV Line Rebuild,Orangeburg #1,False,Cameron Jct,True,46.0,standard_public_gis
7,SC:scrtp_projects.pdf:40:6810 T:1,Cameron Jct – Cameron – St Matthews 46 kV Rebuild,Cameron Jct,True,Cameron,True,46.0,standard_public_gis
8,SC:scrtp_projects.pdf:40:6810 T:2,Cameron Jct – Cameron – St Matthews 46 kV Rebuild,Cameron,True,St Matthews,False,46.0,standard_public_gis
9,SC:scrtp_projects.pdf:48:6809 M:2,Modoc – McCormick 115/46 kV Rebuild,Modoc,True,McCormick,True,46.0,standard_public_gis


## 8. Route low-voltage assets through OSM

Order:

1. exact-voltage OSM graph;
2. if no route, graph that also permits unknown-voltage OSM edges;
3. if the second graph works, mark it `review_required`.

Rows tagged `requires_primary_source` keep that tag even if both endpoints are already independently anchored.


In [14]:
route_rows = []

for _, asset in coverage_gaps.iterrows():
    voltage = float(asset["voltage_kv"])
    voltage_key = int(voltage)

    if voltage_key not in graphs:
        route_rows.append({
            "asset_key": asset["asset_key"],
            "project_name": asset["project_name"],
            "voltage_kv": voltage,
            "route_status": "unsupported_voltage_view",
            "route_method": None,
            "review_note": "No OSM graph view was built for this voltage.",
            "resolution_requirement": asset.get("resolution_requirement"),
            "geometry": None,
        })
        continue

    a = endpoint_lookup(asset.get("endpoint_a"), voltage)
    b = endpoint_lookup(asset.get("endpoint_b"), voltage)

    if not a or not b:
        route_rows.append({
            "asset_key": asset["asset_key"],
            "project_name": asset["project_name"],
            "asset_name": asset.get("asset_name"),
            "endpoint_a": asset.get("endpoint_a"),
            "endpoint_b": asset.get("endpoint_b"),
            "voltage_kv": voltage,
            "route_status": "endpoint_anchor_missing",
            "route_method": None,
            "review_note": "OSM network exists, but one or both endpoint coordinates are still unresolved.",
            "resolution_requirement": asset.get("resolution_requirement"),
            "geometry": None,
        })
        continue

    result = route_between_points(
        graphs[voltage_key],
        a["lon"],
        a["lat"],
        b["lon"],
        b["lat"],
        snap_max_km=2.0,
        max_route_km=500.0,
    )

    if result is None:
        result = route_between_points(
            graphs_review[voltage_key],
            a["lon"],
            a["lat"],
            b["lon"],
            b["lat"],
            snap_max_km=2.0,
            max_route_km=500.0,
        )

    if result is None:
        route_rows.append({
            "asset_key": asset["asset_key"],
            "project_name": asset["project_name"],
            "asset_name": asset.get("asset_name"),
            "endpoint_a": asset.get("endpoint_a"),
            "endpoint_b": asset.get("endpoint_b"),
            "voltage_kv": voltage,
            "route_status": "unresolved",
            "route_method": None,
            "review_note": "Both endpoint locations exist, but no connected OSM path was found within the snap/route limits.",
            "resolution_requirement": asset.get("resolution_requirement"),
            "geometry": None,
        })
        continue

    route_rows.append({
        "asset_key": asset["asset_key"],
        "project_name": asset["project_name"],
        "asset_name": asset.get("asset_name"),
        "endpoint_a": asset.get("endpoint_a"),
        "endpoint_b": asset.get("endpoint_b"),
        "voltage_kv": voltage,
        "route_status": result["route_status"],
        "route_method": result["route_method"],
        "distance_km": result["distance_km"],
        "hops": result["hops"],
        "start_snap_km": result["start_snap_km"],
        "end_snap_km": result["end_snap_km"],
        "osm_way_ids": result["osm_way_ids"],
        "way_names": result["way_names"],
        "operators": result["operators"],
        "unknown_voltage_edge_count": result["unknown_voltage_edge_count"],
        "source": result["source"],
        "attribution": result["attribution"],
        "license": result["license"],
        "resolution_requirement": asset.get("resolution_requirement"),
        "review_note": (
            "Route uses actual OSM line geometry."
            if result["route_status"] == "resolved"
            else "Route uses actual OSM geometry but includes one or more segments with missing OSM voltage metadata."
        ),
        "geometry": result["geometry"],
    })


osm_routes = pd.DataFrame(route_rows)

display(
    osm_routes[
        [
            column
            for column in [
                "project_name",
                "endpoint_a",
                "endpoint_b",
                "voltage_kv",
                "route_status",
                "route_method",
                "distance_km",
                "unknown_voltage_edge_count",
                "resolution_requirement",
                "review_note",
            ]
            if column in osm_routes.columns
        ]
    ]
)


,project_name,endpoint_a,endpoint_b,voltage_kv,route_status,route_method,distance_km,unknown_voltage_edge_count,resolution_requirement,review_note
0,Coit – Gills Creek 115kV: Construct,Coit,Gills Creek,33.0,endpoint_anchor_missing,NaN,NaN,NaN,standard_public_gis,"OSM network exists, but one or both endpoint coordinates are still unresolved."
1,Hooks - Modoc 115/46 kV Rebuild,Hooks,Modoc,46.0,endpoint_anchor_missing,NaN,NaN,NaN,standard_public_gis,"OSM network exists, but one or both endpoint coordinates are still unresolved."
2,Cameron Jct – Elloree 46 kV Rebuild,Cameron Jct,Elloree,46.0,resolved,osm_voltage_graph,15.339134,0.0,standard_public_gis,Route uses actual OSM line geometry.
3,Elloree – Santee City 46 kV: Rebuild,Elloree,Santee City,46.0,endpoint_anchor_missing,NaN,NaN,NaN,standard_public_gis,"OSM network exists, but one or both endpoint coordinates are still unresolved."
4,Urquhart – Aiken PSA 46 kV: Rebuild,Urquhart,Aiken PSA,46.0,endpoint_anchor_missing,NaN,NaN,NaN,standard_public_gis,"OSM network exists, but one or both endpoint coordinates are still unresolved."
5,Adams Run – Red House Road 46 kV: Replace Dawhoo River Crossing and other Structures,Adams Run,Red House Road,46.0,endpoint_anchor_missing,NaN,NaN,NaN,standard_public_gis,"OSM network exists, but one or both endpoint coordinates are still unresolved."
6,Orangeburg #1 - Cameron Jct 46kV Line Rebuild,Orangeburg #1,Cameron Jct,46.0,endpoint_anchor_missing,NaN,NaN,NaN,standard_public_gis,"OSM network exists, but one or both endpoint coordinates are still unresolved."
7,Cameron Jct – Cameron – St Matthews 46 kV Rebuild,Cameron Jct,Cameron,46.0,resolved,osm_voltage_graph,0.000000,0.0,standard_public_gis,Route uses actual OSM line geometry.
8,Cameron Jct – Cameron – St Matthews 46 kV Rebuild,Cameron,St Matthews,46.0,endpoint_anchor_missing,NaN,NaN,NaN,standard_public_gis,"OSM network exists, but one or both endpoint coordinates are still unresolved."
9,Modoc – McCormick 115/46 kV Rebuild,Modoc,McCormick,46.0,unresolved,NaN,NaN,NaN,standard_public_gis,"Both endpoint locations exist, but no connected OSM path was found within the snap/route limits."


## 9. Inspect unlabeled branch/tap nodes

A degree-3+ OSM node can be useful evidence for a small tap that has no distinct facility name.

It is **not automatically the missing Dominion endpoint**. Treat it as a topology candidate that still needs project-source confirmation.


In [15]:
for voltage in [33, 46]:
    tap_candidates = degree_table(graphs[voltage])
    tap_candidates = tap_candidates[
        tap_candidates["degree"] >= 3
    ]

    print(
        f"{voltage} kV degree-3+ nodes:",
        len(tap_candidates),
    )

    display(tap_candidates.head(25))


33 kV degree-3+ nodes: 14


,node_id,degree,lon,lat,way_ids
0,9881486024,4,-81.040563,33.979288,"[1077742260, 1077742261]"
1,9881486025,4,-81.040401,33.979818,"[1077742260, 1077742261]"
2,6088109733,3,-81.057303,34.021081,"[648480645, 1078308097, 1078308098]"
3,9881486012,3,-81.043736,33.976320,"[1077742260, 1078487461]"
4,9885887185,3,-81.052183,34.006092,"[1078308099, 1078308103, 1078308104]"
5,9885887189,3,-81.051091,34.003670,"[1078308099, 1078308100]"
6,9885887197,3,-81.052733,34.003134,"[1078308100, 1078308101]"
7,9887206212,3,-81.047646,34.000016,"[1078487401, 1078487402, 1078487403]"
8,9887222057,3,-81.040932,33.981124,"[1078487403, 1078487405, 1078487406]"
9,9887222899,3,-81.009428,33.972146,"[1078487451, 1078487453]"


46 kV degree-3+ nodes: 117


,node_id,degree,lon,lat,way_ids
0,9788929257,4,-81.240838,32.770375,"[1065939233, 1065939236]"
1,9832244293,4,-81.385849,33.243473,"[1071426805, 1071426814, 1071426817]"
2,9858857948,4,-82.278132,33.887500,"[1074904634, 1074904636, 1074904638]"
3,9864085014,4,-80.867104,33.481209,"[1075548809, 1075548838, 1075548841, 1075548843]"
4,9864085673,4,-80.843564,33.520817,"[1075548848, 1075548850, 1075548862, 1075548863]"
5,1962310286,3,-81.068152,33.294704,"[236469792, 1075801513]"
6,4080229655,3,-81.262526,33.352477,"[405881970, 1071426798, 1071426799]"
7,5169207715,3,-80.577371,33.517902,"[12139895, 1077428685]"
8,5211620240,3,-80.697929,33.546370,"[12139895, 623292013]"
9,9564962200,3,-80.591409,33.527949,"[12139895, 1077428678]"


## 10. Save OSM route audit + GeoJSON

In [16]:
csv_out = osm_routes.copy()

if "geometry" in csv_out.columns:
    csv_out["geometry"] = csv_out["geometry"].apply(
        lambda value: (
            json.dumps(value, separators=(",", ":"))
            if isinstance(value, dict)
            else None
        )
    )

for column in ["osm_way_ids", "way_names", "operators"]:
    if column in csv_out.columns:
        csv_out[column] = csv_out[column].apply(
            lambda value: (
                json.dumps(value)
                if isinstance(value, list)
                else value
            )
        )

csv_out.to_csv(
    OSM_ROUTE_OUTPUT_CSV,
    index=False,
)


features = []

for _, row in osm_routes.iterrows():
    geometry = row.get("geometry")

    if not isinstance(geometry, dict):
        continue

    properties = {
        key: row.get(key)
        for key in [
            "asset_key",
            "project_name",
            "asset_name",
            "endpoint_a",
            "endpoint_b",
            "voltage_kv",
            "route_status",
            "route_method",
            "distance_km",
            "hops",
            "start_snap_km",
            "end_snap_km",
            "osm_way_ids",
            "way_names",
            "operators",
            "unknown_voltage_edge_count",
            "source",
            "attribution",
            "license",
            "resolution_requirement",
            "review_note",
        ]
    }

    for key, value in list(properties.items()):
        if isinstance(value, float) and pd.isna(value):
            properties[key] = None

    features.append({
        "type": "Feature",
        "properties": properties,
        "geometry": geometry,
    })


OSM_ROUTE_OUTPUT_GEOJSON.write_text(
    json.dumps({
        "type": "FeatureCollection",
        "features": features,
    }),
    encoding="utf-8",
)


print("Saved:")
print(" -", TAGGED_ROUTE_RESULTS_CSV)
print(" -", PRIMARY_SOURCE_QUEUE_CSV)
print(" -", OSM_ROUTE_OUTPUT_CSV)
print(" -", OSM_ROUTE_OUTPUT_GEOJSON)
print()
print("OSM routed geometries:", len(features))


Saved:
 - C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\dominion\dominion_route_resolution_v3_tagged.csv
 - C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\dominion\dominion_requires_primary_source.csv
 - C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\dominion\dominion_osm_low_voltage_routes.csv
 - C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\geo\dominion_osm_low_voltage_routes.geojson

OSM routed geometries: 2


## What this proves

If the OSM graph works, the main resolver only needs a small integration step:

```python
route = try_hifld_route(...)

if route is None:
    route = try_osm_route(...)
```

For 33/46 kV, OSM can become the primary graph when HIFLD has no usable network.

Do **not** loosen the fuzzy matcher to chase rows tagged `requires_primary_source`. Those rows should be solved from Dominion project maps, PSC filings, or other primary engineering/regulatory material.
